<a href="https://colab.research.google.com/github/ddoneu/ECON3916-Statistical-Machine-Learning/blob/main/Lab13/%5BLab_13%5D_Hedonic_Pricing_and_the_FWL_Theorem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import matplotlib.pyplot as plt

# Step 1: Ingestion from external source
url = 'https://raw.githubusercontent.com/ddoneu/ECON3916-Statistical-Machine-Learning/refs/heads/main/Data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

In [4]:
df.head()

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54


In [8]:
naive_model = smf.ols("Sale_Price ~ Property_Age", data=df).fit()

print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Wed, 18 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        19:55:22   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [11]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])


                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Wed, 18 Mar 2026   Prob (F-statistic):               0.00
Time:                        19:59:32   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [16]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

df['Price_Residuals']

,Price_Residuals
0,-56823.332740
1,19000.990249
2,-69149.815200
3,18481.157582
4,-2619.815668
...,...
995,21560.763160
996,-1940.516556
997,-3398.817768
998,2026.560249


In [17]:
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

df['Age_Residuals']

,Age_Residuals
0,0.621803
1,-13.689856
2,3.233510
3,5.347789
4,4.053610
...,...
995,-14.814575
996,-6.511286
997,-1.986001
998,-4.589856


In [19]:
# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals - 1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])


FWL Isolated Age Coefficient: -2063.129216802139


# Task
Create an interactive 3D scatter plot showing 'Sale_Price' against 'Property_Age' and 'Distance_to_Tech_Hub', overlaying a regression hyperplane based on the `multi_model`. Also, provide an interpretation of this hyperplane in the context of controlling for confounding variables.

## Prepare Data for 3D Plotting

### Subtask:
Generate a meshgrid for 'Property_Age' and 'Distance_to_Tech_Hub' across their respective ranges. Then, use the previously fitted 'multi_model' to predict 'Sale_Price' for each point on this meshgrid to create the data for the regression hyperplane. This step will also extract coefficients from the 'multi_model' to be used in the prediction.


**Reasoning**:
I will generate a meshgrid for the independent variables, predict the 'Sale_Price' using the `multi_model`, and extract the model coefficients, all in one step to prepare the data for 3D plotting of the regression hyperplane.



In [20]:
import numpy as np
import pandas as pd

# 1. Determine min and max values for 'Property_Age' and 'Distance_to_Tech_Hub'
min_age, max_age = df['Property_Age'].min(), df['Property_Age'].max()
min_distance, max_distance = df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max()

# 2. Create 1D arrays for each variable
prop_age_grid = np.linspace(min_age, max_age, 100) # Using 100 points for smoother hyperplane
distance_grid = np.linspace(min_distance, max_distance, 100) # Using 100 points for smoother hyperplane

# 3. Use np.meshgrid to create a 2D meshgrid
X_age, X_distance = np.meshgrid(prop_age_grid, distance_grid)

# 4. Create a new DataFrame from the meshgrid variables
plot_df = pd.DataFrame({
    'Property_Age': X_age.ravel(),
    'Distance_to_Tech_Hub': X_distance.ravel()
})

# 5. Use the multi_model.predict() method on this new DataFrame
predicted_sale_price = multi_model.predict(plot_df)

# Reshape the predicted values to match the meshgrid for plotting
Z_price = predicted_sale_price.values.reshape(X_age.shape)

# 6. Extract the coefficients from the multi_model's params attribute
intercept = multi_model.params['Intercept']
coeff_age = multi_model.params['Property_Age']
coeff_distance = multi_model.params['Distance_to_Tech_Hub']

print(f"Intercept: {intercept:.2f}")
print(f"Coefficient for Property_Age: {coeff_age:.2f}")
print(f"Coefficient for Distance_to_Tech_Hub: {coeff_distance:.2f}")
print("Meshgrid and predicted Sale_Price for hyperplane created, and coefficients extracted.")

Intercept: 1202971.27
Coefficient for Property_Age: -2063.13
Coefficient for Distance_to_Tech_Hub: -7964.24
Meshgrid and predicted Sale_Price for hyperplane created, and coefficients extracted.


## Create Interactive 3D Scatter Plot with Regression Plane

### Subtask:
Generate an interactive 3D scatter plot using `plotly.graph_objects`. This plot will visualize the original 'Sale_Price' data points against 'Property_Age' and 'Distance_to_Tech_Hub'. Overlay the generated regression hyperplane on this scatter plot. Ensure to add inline comments explaining the meshgrid generation and how coefficients are extracted from the `statsmodels` results for the hyperplane calculation.


**Reasoning**:
I will generate an interactive 3D scatter plot using `plotly.graph_objects` to visualize the original 'Sale_Price' data points against 'Property_Age' and 'Distance_to_Tech_Hub', and overlay the generated regression hyperplane based on the `multi_model`. This directly addresses the current subtask.



In [21]:
import plotly.graph_objects as go

# Create a Scatter3d trace for the original data points
scatter_trace = go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'],
    mode='markers',
    marker=dict(
        size=4,
        opacity=0.8,
        color='blue' # Original data points
    ),
    name='Actual Sale Price'
)

# Create a Surface trace for the regression hyperplane
surface_trace = go.Surface(
    x=X_age,
    y=X_distance,
    z=Z_price,
    opacity=0.6,
    colorscale='Viridis', # Color scale for the surface
    name='Regression Hyperplane'
)

# Create the Figure object and add traces
fig = go.Figure(data=[scatter_trace, surface_trace])

# Update layout with title and axis labels
fig.update_layout(
    title='3D Scatter Plot with Regression Hyperplane',
    scene=dict(
        xaxis_title='Property Age',
        yaxis_title='Distance to Tech Hub',
        zaxis_title='Sale Price'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Display the figure
fig.show()

### Interpretation of the Regression Hyperplane

The 3D interactive plot visualizes the relationship between `Sale_Price`, `Property_Age`, and `Distance_to_Tech_Hub`. The original data points represent the actual sale prices, while the overlaid regression hyperplane represents the predicted `Sale_Price` values from our `multi_model`.

**Controlling for Confounding Variables:**

In this multivariate regression (`Sale_Price ~ Property_Age + Distance_to_Tech_Hub`), the hyperplane demonstrates the *conditional* relationship between `Sale_Price` and each independent variable, holding the other constant. Specifically:

- The slope of the hyperplane along the 'Property Age' axis, while moving parallel to the 'Distance to Tech Hub' axis, represents the effect of `Property_Age` on `Sale_Price` *after accounting for* `Distance_to_Tech_Hub`. The `multi_model`'s coefficient for `Property_Age` was **-2063.13**, indicating that, for every additional year of property age, the sale price is predicted to decrease by approximately $2063, *assuming the distance to the tech hub remains constant*.

- Similarly, the slope along the 'Distance to Tech Hub' axis, moving parallel to the 'Property Age' axis, represents the effect of `Distance_to_Tech_Hub` on `Sale_Price` *after accounting for* `Property_Age`. The `multi_model`'s coefficient for `Distance_to_Tech_Hub` was **-7964.24**, suggesting that for every additional unit of distance to a tech hub, the sale price is predicted to decrease by approximately $7964, *assuming the property age remains constant*.

This is a crucial aspect of multivariate regression: by including multiple independent variables, we are effectively "controlling for" or "holding constant" the effects of the other variables when interpreting a specific coefficient. Without this control, the estimated effect of `Property_Age` (as seen in the naive model where its coefficient was positive: **5573.63**) would be confounded by its correlation with `Distance_to_Tech_Hub`, leading to a potentially misleading interpretation. The multivariate model, and thus the hyperplane, provides a more accurate representation of the isolated impact of each variable on `Sale_Price`.

## Final Task

### Subtask:
The 3D interactive plot with the regression hyperplane has been generated and an interpretation of the hyperplane has been provided.


## Summary:

### Q&A
The regression hyperplane's interpretation reveals that controlling for confounding variables is crucial. The `multi_model` shows that for every additional year of property age, the sale price is predicted to decrease by approximately \$2063, assuming the distance to the tech hub remains constant. Similarly, for every additional unit of distance to a tech hub, the sale price is predicted to decrease by approximately \$7964, assuming the property age remains constant. This contrasts with a simpler, "naive" model where the coefficient for `Property_Age` might have been misleadingly positive (5573.63), demonstrating the importance of multivariate analysis in isolating variable effects.

### Data Analysis Key Findings
*   The `multi_model` yielded an intercept of \$1,202,971.27, indicating the baseline `Sale_Price` when `Property_Age` and `Distance_to_Tech_Hub` are zero.
*   The coefficient for `Property_Age` was -2063.13, suggesting that for each additional year of property age, `Sale_Price` is predicted to decrease by approximately \$2063, holding `Distance_to_Tech_Hub` constant.
*   The coefficient for `Distance_to_Tech_Hub` was -7964.24, implying that for each additional unit of distance to a tech hub, `Sale_Price` is predicted to decrease by approximately \$7964, holding `Property_Age` constant.
*   An interactive 3D scatter plot was successfully generated, visualizing the actual `Sale_Price` data points alongside a regression hyperplane that represents the predicted `Sale_Price` based on `Property_Age` and `Distance_to_Tech_Hub`.
*   The hyperplane effectively demonstrates the conditional relationships between `Sale_Price` and the independent variables, showcasing how multivariate regression "controls for" other variables to provide a more accurate interpretation of individual effects.

### Insights or Next Steps
*   The significant negative coefficients for both `Property_Age` and `Distance_to_Tech_Hub` highlight these as key depreciating factors for property value, even when accounting for their interdependencies. This insight can be crucial for property valuation and investment strategies.
*   Further analysis could explore non-linear relationships or interaction effects between `Property_Age` and `Distance_to_Tech_Hub`, or incorporate additional potential confounding variables (e.g., property size, number of bedrooms, neighborhood amenities) to build a more comprehensive predictive model.
